# OmegAMP in 20 minutes

You are going to ask a model to write **antimicrobial peptides** (AMPs): short amino-acid sequences that can kill bacteria. You do not need a biology background.

Every code cell is ready to run. Change a value only on a line marked `# change this`.

**How to use this notebook**

1. Run the Installation section once. If conda is available, it creates the `public-omegamp` environment; otherwise it installs into this environment. After a conda install, switch to the `public-omegamp` kernel (Kernel → Change kernel) before continuing.
2. Run cells from top to bottom with Shift+Enter.
3. The cell that loads the model takes **1–3 minutes**. Leave it running.
4. Later cells design **one** peptide each. About a minute per cell is normal.
5. Two cells are marked **One minute with your neighbor**. Those are the only pair discussions.

**The two choices every OmegAMP call makes**

| Choice | Values you will use | In plain words |
| --- | --- | --- |
| Generation mode | `de-novo`, `analog`, `motif` | Start from scratch, start near a known peptide, or lock some letters |
| Conditioning | `targeted`, `unconditional` | Ask for a length and a charge, or let those properties vary |

The call is always `generate_samples.main(...)`. Sections below only change its arguments.

## Installation

Run this cell once. It clones [OmegAMP](https://github.com/szczurek-lab/OmegAMP) when this folder is not already the repository, then installs the project the same way as the README.

- **Conda is available.** The cell creates the `public-omegamp` environment with Python 3.11 and installs OmegAMP there. When it finishes, switch the kernel to `public-omegamp`, then continue. Later cells need that kernel.
- **Conda is not available.** The cell installs OmegAMP into the environment of this notebook. Keep this kernel and continue.

It also downloads the generative-model checkpoint and embeddings when those files are not already on disk.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

repo_url = "https://github.com/szczurek-lab/OmegAMP.git"

if Path("pyproject.toml").exists():
    print("Already inside the OmegAMP folder. Skipping clone.")
else:
    repo = Path("OmegAMP")
    if not (repo / "pyproject.toml").exists():
        subprocess.check_call(["git", "clone", repo_url, str(repo)])
    os.chdir(repo)
    print("Working directory:", Path(".").resolve())

ckpt = Path("models/generative_model.ckpt")
embeddings = Path("data/generative-model-data/generative-model-embeddings.h5")

if shutil.which("conda"):
    listed = subprocess.run(
        ["conda", "env", "list"],
        check=True,
        capture_output=True,
        text=True,
    )
    exists = any(
        line.split()[:1] == ["public-omegamp"] for line in listed.stdout.splitlines()
    )
    if not exists:
        subprocess.check_call(
            ["conda", "create", "-n", "public-omegamp", "python=3.11", "-y"]
        )
    subprocess.check_call(
        ["conda", "run", "-n", "public-omegamp", "pip", "install", "-e", "."]
    )
    subprocess.check_call(
        [
            "conda",
            "run",
            "-n",
            "public-omegamp",
            "python",
            "-m",
            "ipykernel",
            "install",
            "--user",
            "--name",
            "public-omegamp",
            "--display-name",
            "public-omegamp",
        ]
    )
    python = ["conda", "run", "-n", "public-omegamp", "python", "-m"]
    installed_with_conda = True
    print("Installed into the conda environment public-omegamp.")
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "."])
    python = [sys.executable, "-m"]
    installed_with_conda = False
    print("Conda was not found. Installed into this environment.")

ckpt.parent.mkdir(parents=True, exist_ok=True)
embeddings.parent.mkdir(parents=True, exist_ok=True)
if not ckpt.exists():
    subprocess.check_call(
        python
        + [
            "gdown",
            "https://drive.google.com/uc?id=1KO-6Aa7K5_G03DTiwfCa-gPAIOsuxigd",
            "-O",
            str(ckpt),
        ]
    )
if not embeddings.exists():
    subprocess.check_call(
        python
        + [
            "gdown",
            "https://drive.google.com/uc?id=1divlvNxsmjYacqb7wb6nK06b8XF8JlOw",
            "-O",
            str(embeddings),
        ]
    )
print("Installation finished.")
if installed_with_conda:
    print(
        "Switch the kernel to public-omegamp (Kernel \u2192 Change kernel), "
        "then continue from the next section."
    )

**Switch the kernel before the next section** if the cell used conda. Choose `public-omegamp` under Kernel → Change kernel. OmegAMP was installed into that environment, and the kernel that ran the cell above does not see it yet.

On Google Colab there is no conda and no kernel list. The cell installs into the current runtime, so stay on that runtime and continue.

## 1. Load the model once

Run this cell and wait for `Model is ready`. The same `model` object is reused below, so you do not load it again.

In [ ]:
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

from project.config import load_model_for_inference
from project.scripts.inference import generate_samples

if not Path("models/generative_model.ckpt").exists():
    raise FileNotFoundError(
        "Checkpoint not found. Open this notebook from the OmegAMP folder "
        "(the folder that contains models/ and config/)."
    )

if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()

with initialize_config_dir(version_base=None, config_dir=str(Path("config").resolve())):
    config = compose(config_name="train")

model = load_model_for_inference(config, "models/generative_model.ckpt")
print("Model is ready. Continue to the next cell.")

Many AMPs carry a **positive charge**, which helps them meet bacterial membranes. The main positive letters are **K** (lysine) and **R** (arginine). The next section counts those letters. The count is a quick look, and it does not have to equal the charge you request.

## 2. Design one peptide from scratch

`de-novo` starts from noise. `targeted` asks for length 20 and charge 6. `hydrophobicity="-"` leaves hydrophobicity unspecified, so the model samples it.

`seed=7` fixes the random draw. Anyone who keeps these settings gets the same peptide.

In [ ]:
sequences, _ = generate_samples.main(
    generation_mode="de-novo",
    conditioning_strategy="targeted",
    length="20",
    charge="6",
    hydrophobicity="-",
    num_samples=1,
    batch_size=1,
    seed=7,
    model=model,
    output_fasta=None,
    conditioning_output_path=None,
)

peptide = sequences[0]
print(peptide)
print("length:", len(peptide))
print("K or R letters:", sum(aa in "KR" for aa in peptide))

With `seed=7` the printout is `PVHQRPAKLRRNIIVHVSPR`: length 20, and 5 K or R letters. The request was charge 6. K+R can sit near that number without equaling it, because other letters also affect charge.

## 3. Ask for a lower charge

Same call as above. The only edit is `charge`.

Property values use one of three forms:

| You type | Meaning |
| --- | --- |
| `"-"` | the model chooses |
| `"6"` | that exact value |
| `"4:8"` | a value between 4 and 8 |

In [ ]:
charge = "2"  # change this: "2", "6", or "4:8"

gentler, _ = generate_samples.main(
    generation_mode="de-novo",
    conditioning_strategy="targeted",
    length="20",
    charge=charge,
    hydrophobicity="-",
    num_samples=1,
    batch_size=1,
    seed=7,
    model=model,
    output_fasta=None,
    conditioning_output_path=None,
)

gentle = gentler[0]
print("charge 6:", peptide, "  K+R =", sum(aa in "KR" for aa in peptide))
print(f"charge {charge}:", gentle, "  K+R =", sum(aa in "KR" for aa in gentle))

### One minute with your neighbor

With `seed=7`, charge 6 gives `PVHQRPAKLRRNIIVHVSPR` (K+R = 5) and charge 2 gives `PVHDRPAMLTRNILVHVTPV` (K+R = 2). Both stay length 20. The lower charge request came with fewer K and R letters, and the rest of the sequence still changed.

Why can K+R drop from 5 to 2 while the length stays 20? Charge is one constraint. The other positions are still free.

## 4. Start from a known peptide

**Analog** mode starts from a peptide you provide, noises it, and denoises. The sequence below is from the curated non-AMP set (`KIAGKIAKKAGKIAKIA`). `unconditional` fixes the AMP label and leaves length and charge free.

`tau` is the fraction of the 1000-step trajectory that is rerun. `0.3` reruns the last 300 steps. With this peptide and `seed=7`, the result is `KIKGKIGKKGVKPAKVV`: still 17 letters, with 10 positions unchanged and 7 rewritten.

The cell writes a one-line FASTA file. With a FASTA input, OmegAMP returns one peptide per entry, and `num_samples` is not what sets the count.

In [ ]:
original = "KIAGKIAKKAGKIAKIA"
demo_dir = Path("class-demo-inputs")
demo_dir.mkdir(exist_ok=True)
(demo_dir / "one-analog.fasta").write_text(f">demo\n{original}\n")

analogs, _ = generate_samples.main(
    generation_mode="analog",
    conditioning_strategy="unconditional",
    analog_sequences="class-demo-inputs/one-analog.fasta",
    tau=0.3,  # change this: 0.3 rewrites some letters, 0.5 wanders further
    num_samples=1,
    batch_size=1,
    seed=7,
    model=model,
    output_fasta=None,
    conditioning_output_path=None,
)

designed = analogs[0]
print("original:", original)
print("designed: ", designed)
shared = sum(a == b for a, b in zip(original, designed))
print(f"positions still equal: {shared} of {len(original)}")

### One minute with your neighbor

With `tau=0.3` and `seed=7`, `KIAGKIAKKAGKIAKIA` becomes `KIKGKIGKKGVKPAKVV`. Ten positions still match, and seven do not. The peptide moved, and much of it stayed.

When would you want fewer changes than this, and when would you want more? A smaller `tau` reruns fewer steps. A larger `tau` can rewrite more letters. This notebook does not score whether the result is active.

## 5. Lock some letters (if you still have time)

**Motif** mode keeps the letters you write and fills each `-`. In `KIAGKIAKK--------`, the first nine letters are locked and the last eight are open. Length is not fixed here, so the peptide can grow past the motif.

Skip this section if the session is ending. Sections 2 and 4 already used the main call. The cell prints a position only when a locked letter is not kept. With `seed=7` that line does not appear: the design is `KIAGKIAKKRNVIVHVSPRY` (length 20), and it still begins with `KIAGKIAKK`.

In [ ]:
motif = "KIAGKIAKK--------"  # change this: letters stay, "-" is designed
demo_dir = Path("class-demo-inputs")
demo_dir.mkdir(exist_ok=True)
(demo_dir / "one-motif.fasta").write_text(f">demo\n{motif}\n")

filled, _ = generate_samples.main(
    generation_mode="motif",
    conditioning_strategy="unconditional",
    motif_sequences="class-demo-inputs/one-motif.fasta",
    num_samples=1,
    batch_size=1,
    seed=7,
    model=model,
    output_fasta=None,
    conditioning_output_path=None,
)

print("motif:    ", motif)
print("designed: ", filled[0])
for position, (locked, got) in enumerate(zip(motif, filled[0]), start=1):
    if locked != "-" and locked != got:
        print(f"position {position}: requested {locked}, got {got}")

With `seed=7`, no mismatch line is printed: the locked prefix `KIAGKIAKK` is kept. The peptide is 20 letters, longer than the 17-character motif, because length was left free. A line such as `position 3: requested K, got A` would appear only if a locked letter changed.

## What you ran

| Section | Arguments | Name used in the OmegAMP paper |
| --- | --- | --- |
| 2 and 3 | `de-novo` + `targeted` | OmegAMP-DT |
| 4 | `analog` + `unconditional` | OmegAMP-AU |
| 5 | `motif` + `unconditional` | OmegAMP-MU |

The same function accepts other pairs. Analog generation plus a charge target is OmegAMP-AT: set `generation_mode="analog"` and `conditioning_strategy="targeted"` together, and pass both `analog_sequences` and `charge`.

A third conditioning mode, `prototype-derived`, copies length, charge, and hydrophobicity from a known peptide instead of typing those numbers yourself. It uses `prototype_sequences` and `sigma` (how far those properties may drift).

More combinations are listed in `README-Inference.md`.